# Efficient LR-QAOA vs WalkSAT benchmark (BM24)

Rigorous comparison aligned with `Final OG LR QAOA vs QAOA vs walksat.ipynb`, but faster than `sweep_lr_depth_until_win.py`:

1. **One** SAT benchmark dataset with precomputed `H_diag` (shared across depths).
2. **WalkSAT + WalkSATlm once** (Numba), not repeated every depth.
3. **Per depth:** train `(dg, db)` + LR-QAOA only (`bm24_qaoa_sim.run_qaoa`).
4. Same metrics: median `1/p_succ` vs `n` → log₂ scaling exponent; plot vs depth.

Uses BM24 half-angle convention via `train_lr_notebook_protocol` + `make_lr_angles`.

In [ ]:
# --- Configuration (edit here) ---
from pathlib import Path

K = 8
R = 176.54
SEED = 27

TRAIN_N = 12
TRAIN_SIZE = 50

N_MIN, N_MAX = 9, 16          # notebook-like window; use 10,12 for a quick smoke test
TEST_SIZE = 50                # instances per n

DEPTH_MIN, DEPTH_MAX = 2, 20  # sweep range
SKIP_GRID = False             # True = notebook deep-sweep style (COBYLA only)
COBYLA_MAXITER = 200
LR_BETA_SCHEDULE = "decreasing"

P_NOISE = 0.5                 # both WalkSAT and WalkSATlm (notebook default)
MAX_FLIPS = 100_000
WALKSATLM_W1, WALKSATLM_W2 = 6, 5

STOP_ON_WIN = False           # True: stop when LR log2 slope beats both classical lines

OUTPUT_DIR = Path("bm24_runs")  # relative to phasecraft/ when cwd is phasecraft

In [ ]:
import sys
from pathlib import Path

PHASECRAFT = Path.cwd() if (Path.cwd() / "bm24_qaoa_sim.py").is_file() else Path.cwd() / "phasecraft"
if str(PHASECRAFT) not in sys.path:
    sys.path.insert(0, str(PHASECRAFT))

import numpy as np

from train_lr_notebook_protocol import generate_training_h_diagonals
from lr_benchmark_helpers import (
    generate_benchmark_dataset_cached,
    evaluate_classical_baselines_once,
    run_efficient_depth_sweep,
    HAS_NUMBA,
)

print(f"phasecraft dir: {PHASECRAFT.resolve()}")
print(f"Numba available: {HAS_NUMBA}")

## 1. Build datasets (once)

In [ ]:
print("Training set (SAT-filtered H_diag at n=TRAIN_N)...")
training_h = generate_training_h_diagonals(
    train_n=TRAIN_N,
    k=K,
    r=R,
    train_size=TRAIN_SIZE,
    base_seed=SEED,
    m_sampling="notebook",
)
print(f"  {len(training_h)} instances")

n_values = list(range(int(N_MIN), int(N_MAX) + 1))
print(f"Benchmark set n={n_values}, test_size={TEST_SIZE}...")
dataset = generate_benchmark_dataset_cached(
    n_values=n_values,
    k=K,
    r=R,
    test_size=TEST_SIZE,
    base_seed=SEED,
)
print("  done.")

## 2. Classical baselines (once per dataset)

In [ ]:
classical = evaluate_classical_baselines_once(
    dataset,
    k=K,
    base_seed=SEED,
    max_flips=MAX_FLIPS,
    p_noise=P_NOISE,
    walksatlm_w1=WALKSATLM_W1,
    walksatlm_w2=WALKSATLM_W2,
    use_numba_walksatlm=True,
)
ws = classical["fitted_exponents_log2"]["walksat"]
lm = classical["fitted_exponents_log2"]["walksatlm"]
print(f"Classical scaling (log2 slope of median flips vs n):")
print(f"  WalkSAT   = {ws:.4f}")
print(f"  WalkSATlm = {lm:.4f}")

## 3. Depth sweep: train LR + benchmark QAOA only

In [ ]:
trace, plot_path = run_efficient_depth_sweep(
    depth_min=DEPTH_MIN,
    depth_max=DEPTH_MAX,
    n_min=N_MIN,
    n_max=N_MAX,
    k=K,
    r=R,
    test_size=TEST_SIZE,
    train_n=TRAIN_N,
    train_size=TRAIN_SIZE,
    base_seed=SEED,
    training_h=training_h,
    dataset=dataset,
    classical=classical,
    skip_grid=SKIP_GRID,
    cobyla_maxiter=COBYLA_MAXITER,
    lr_beta_schedule=LR_BETA_SCHEDULE,
    stop_on_win=STOP_ON_WIN,
    output_dir=PHASECRAFT / OUTPUT_DIR,
)
plot_path

## 4. Inspect trace

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        "p": row["depth"],
        "dg": row["delta_gamma"],
        "db": row["delta_beta"],
        "lr_log2": row["lr_log2_slope"],
        "beats_both": row["beats_both_on_scaling"],
        "elapsed_s": row["elapsed_s"],
    }
    for row in trace
])
df

## Optional: single-depth diagnostic plot

Reuses saved angles from the trace; compares median `1/p` vs median flips at each `n`.

In [ ]:
from bm24_qaoa_sim import make_lr_angles, plot_benchmark_comparison

DEPTH_PLOT = trace[-1]["depth"] if trace else DEPTH_MIN
row = next(r for r in trace if r["depth"] == DEPTH_PLOT)
betas, gammas = make_lr_angles(
    row["delta_gamma"], row["delta_beta"], int(DEPTH_PLOT),
    beta_schedule=LR_BETA_SCHEDULE, angle_convention="bm24",
)

# Build minimal benchmark JSON for plotting helper
lr_pn = row["lr_per_n"]
ns = sorted(int(k) for k in lr_pn.keys())
bres = {
    "settings": {"k": K, "r": R, "depth": DEPTH_PLOT, "require_sat": True},
    "results": {
        "lr_qaoa": {
            "per_n": {
                str(n): {
                    "median_runtime": lr_pn[str(n)]["median_runtime"],
                    "median_success": lr_pn[str(n)]["median_success"],
                    "mean_success": lr_pn[str(n)]["median_success"],
                }
                for n in ns
            }
        },
        "walksat": {
            "per_n": {
                str(n): {"median_flips": classical["per_n"][n]["median_walksat"]}
                for n in ns
            }
        },
        "walksatlm": {
            "per_n": {
                str(n): {"median_flips": classical["per_n"][n]["median_walksatlm"]}
                for n in ns
            }
        },
    },
}
diag_png = (PHASECRAFT / OUTPUT_DIR / f"efficient-p{DEPTH_PLOT}-diag.png").resolve()
plot_benchmark_comparison(bres, diag_png, equiv_flips_per_shot=1.0)
diag_png